# Task 2a: Corruption Classifier Optuna Search

Hyperparameter tuning for the Task 2a CNN classifier (Plan 6 §2a.5 & Plan 7).
Searches learning rate, batch size, channel depth, dropout, and weight decay.

In [ ]:
# 1. Config
N_TRIALS = 20
OPTUNA_EPOCHS = 15
FULL_TRAIN_EPOCHS = 35
STUDY_NAME = "task2a_classifier"
STORAGE_DB = "sqlite:///task2a_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_classifier import CorruptionClassifier
from src.train_loop import train_one_epoch_classifier, validate_classifier, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device()

In [ ]:
# 3. Data Loading
train_dataset = PetDataset(mode="train", corruption_mode="label", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="label", seed=SEED)

In [ ]:
# 4. Optuna Objective
def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    base_channels = trial.suggest_categorical("base_channels", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = CorruptionClassifier(
        base_channels=base_channels,
        dropout=dropout,
        num_classes=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_acc = 0.0

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_classifier(model, train_loader, optimizer, device)
        val_res = validate_classifier(model, val_loader, device)
        acc = val_res["val_acc"]
        if acc > best_acc:
            best_acc = acc
        trial.report(acc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_acc

In [ ]:
# 5. Run Optimization
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

print(f"Best Trial: #{study.best_trial.number} with Accuracy: {study.best_value:.2%}")
print(study.best_params)

In [ ]:
# 6. Save Best Config & Retrain
best_config_path = CONFIGS_ROOT / "task2a_best_config.yaml"
with open(best_config_path, "w") as f:
    yaml.dump({
        "task": "task2a_corruption_classifier",
        "best_trial_number": int(study.best_trial.number),
        "best_accuracy": float(study.best_value),
        "params": study.best_params,
    }, f, indent=2)

print(f"Saved to {best_config_path}")